In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import (
    KaplanMeierFitter,
    CoxPHFitter
)

from lifelines.statistics import (
    proportional_hazard_test,
    logrank_test
)   

In [ ]:
df = pd.read_csv("../combination/totalData.csv", sep=";")

In [ ]:
def base_survival_analysis(
    df,
    duration_col="remainingDays",
    event_col="eventHasHappend",
    feature_columns=None,
    km_group_column=None,
    show_plots=True,
    penalizer=0.01
):
    """
    Perform a basic survival-analysis workflow.

    Includes:
        1. Dataset / event summary
        2. Overall Kaplan-Meier estimator
        3. Optional Kaplan-Meier comparison between groups
        4. Log-rank test
        5. Univariate Cox-PH models
        6. Multivariate Cox-PH model
        7. Proportional-hazards tests
        8. Concordance index

    Parameters
    ----------
    df : pd.DataFrame
        Survival dataset.

    duration_col : str
        Survival duration column.

    event_col : str
        Event indicator. True/1 = event occurred.

    feature_columns : list[str] or None
        Features to include in Cox analysis.

        If None, suitable numerical features are selected automatically.

    km_group_column : str or None
        Optional categorical/binary variable for Kaplan-Meier comparison.

    show_plots : bool
        Display Kaplan-Meier plots.

    penalizer : float
        L2 penalization for Cox-PH.
        Useful with many correlated predictors.

    Returns
    -------
    dict containing:
        dataset_summary
        km_model
        km_group_results
        univariate_cox
        multivariate_cox
        multivariate_summary
        ph_test
    """

    data = df.copy()

    # ==========================================================
    # 1. Prepare outcome
    # ==========================================================

    data[duration_col] = pd.to_numeric(
        data[duration_col],
        errors="coerce"
    )

    data[event_col] = (
        data[event_col]
        .astype(bool)
        .astype(int)
    )

    data = data[
        data[duration_col].notna()
        & (data[duration_col] >= 0)
        & data[event_col].notna()
    ].copy()

    # ==========================================================
    # 2. Dataset summary
    # ==========================================================

    dataset_summary = pd.DataFrame({
        "metric": [
            "Observations",
            "Events",
            "Censored",
            "Event rate (%)",
            "Median duration",
            "Mean duration",
            "Minimum duration",
            "Maximum duration"
        ],

        "value": [
            len(data),
            data[event_col].sum(),
            (data[event_col] == 0).sum(),
            data[event_col].mean() * 100,
            data[duration_col].median(),
            data[duration_col].mean(),
            data[duration_col].min(),
            data[duration_col].max()
        ]
    })

    print("\n=== Dataset summary ===")
    print(dataset_summary)

    # ==========================================================
    # 3. Overall Kaplan-Meier
    # ==========================================================

    kmf = KaplanMeierFitter()

    kmf.fit(
        durations=data[duration_col],
        event_observed=data[event_col],
        label="Overall"
    )

    print("\n=== Kaplan-Meier ===")

    print(
        "Median survival time:",
        kmf.median_survival_time_
    )

    if show_plots:

        fig, ax = plt.subplots(
            figsize=(9, 5)
        )

        kmf.plot_survival_function(
            ax=ax,
            ci_show=True
        )

        ax.set_title(
            "Kaplan-Meier Survival Estimate"
        )

        ax.set_xlabel(
            "Remaining Days"
        )

        ax.set_ylabel(
            "Survival Probability"
        )

        ax.set_ylim(0, 1.05)

        fig.tight_layout()
        plt.show()

    # ==========================================================
    # 4. Optional grouped Kaplan-Meier
    # ==========================================================

    km_group_results = None

    if (
        km_group_column is not None
        and km_group_column in data.columns
    ):

        group_data = data[
            [
                duration_col,
                event_col,
                km_group_column
            ]
        ].dropna()

        groups = (
            group_data[km_group_column]
            .unique()
        )

        if len(groups) >= 2:

            if show_plots:

                fig, ax = plt.subplots(
                    figsize=(9, 5)
                )

            group_models = {}

            for group in groups:

                mask = (
                    group_data[km_group_column]
                    == group
                )

                group_kmf = (
                    KaplanMeierFitter()
                )

                group_kmf.fit(
                    group_data.loc[
                        mask,
                        duration_col
                    ],
                    event_observed=
                    group_data.loc[
                        mask,
                        event_col
                    ],
                    label=str(group)
                )

                group_models[group] = (
                    group_kmf
                )

                if show_plots:

                    group_kmf.plot_survival_function(
                        ax=ax,
                        ci_show=True
                    )

            if show_plots:

                ax.set_title(
                    f"Kaplan-Meier by {km_group_column}"
                )

                ax.set_xlabel(
                    "Remaining Days"
                )

                ax.set_ylabel(
                    "Survival Probability"
                )

                ax.set_ylim(
                    0,
                    1.05
                )

                fig.tight_layout()
                plt.show()

            # ----------------------------------------------
            # Log-rank for exactly two groups
            # ----------------------------------------------

            logrank = None

            if len(groups) == 2:

                g1 = groups[0]
                g2 = groups[1]

                mask1 = (
                    group_data[km_group_column]
                    == g1
                )

                mask2 = (
                    group_data[km_group_column]
                    == g2
                )

                logrank = logrank_test(
                    group_data.loc[
                        mask1,
                        duration_col
                    ],

                    group_data.loc[
                        mask2,
                        duration_col
                    ],

                    event_observed_A=
                    group_data.loc[
                        mask1,
                        event_col
                    ],

                    event_observed_B=
                    group_data.loc[
                        mask2,
                        event_col
                    ]
                )

                print(
                    f"\nLog-rank test: "
                    f"{g1} vs {g2}"
                )

                print(
                    "p-value:",
                    logrank.p_value
                )

            km_group_results = {
                "models":
                    group_models,

                "logrank":
                    logrank
            }

    # ==========================================================
    # 5. Select Cox features
    # ==========================================================

    # These should NOT enter the model
    leakage_columns = [
        "id",

        # Outcome
        duration_col,
        event_col,

        # Exact future endpoint
        "targetEndDate",

        # Future-information count
        "assignmentsAfterCut",

        # Actual future caregiver exit
        "ausgesch-am",

        # Raw dates / identifiers
        "cutDate",
        "startofCaregiver",
        "endofCaregiver",
        "eingestellt-am"
    ]

    if feature_columns is None:

        feature_columns = (
            data
            .select_dtypes(
                include=["number", "bool"]
            )
            .columns
            .difference(leakage_columns)
            .tolist()
        )

    else:

        feature_columns = [
            col
            for col in feature_columns
            if col in data.columns
            and col not in leakage_columns
        ]

    # ==========================================================
    # 6. UNIVARIATE COX-PH
    # ==========================================================

    univariate_results = []

    for feature in feature_columns:

        cox_data = data[
            [
                duration_col,
                event_col,
                feature
            ]
        ].copy()

        cox_data[feature] = pd.to_numeric(
            cox_data[feature],
            errors="coerce"
        )

        cox_data = cox_data.dropna()

        # Need variation
        if cox_data[feature].nunique() < 2:
            continue

        # Need enough observations
        if len(cox_data) < 10:
            continue

        try:

            cph = CoxPHFitter(
                penalizer=penalizer
            )

            cph.fit(
                cox_data,
                duration_col=duration_col,
                event_col=event_col
            )

            row = cph.summary.loc[
                feature
            ]

            univariate_results.append({
                "feature":
                    feature,

                "coef":
                    row["coef"],

                "hazard_ratio":
                    row["exp(coef)"],

                "ci_lower":
                    row[
                        "exp(coef) lower 95%"
                    ],

                "ci_upper":
                    row[
                        "exp(coef) upper 95%"
                    ],

                "p":
                    row["p"],

                "n":
                    len(cox_data)
            })

        except Exception as e:

            print(
                f"Skipped {feature}: {e}"
            )

    univariate_cox = (
        pd.DataFrame(
            univariate_results
        )
    )

    if not univariate_cox.empty:

        univariate_cox[
            "abs_coef"
        ] = (
            univariate_cox[
                "coef"
            ].abs()
        )

        univariate_cox = (
            univariate_cox
            .sort_values(
                "p"
            )
            .reset_index(
                drop=True
            )
        )

    # ==========================================================
    # 7. MULTIVARIATE COX-PH
    # ==========================================================

    multivariate_cox = None
    multivariate_summary = None
    ph_test_df = None

    selected_features = [
        col
        for col in feature_columns
        if col in data.columns
    ]

    if selected_features:

        cox_data = data[
            [duration_col, event_col]
            + selected_features
        ].copy()

        # Convert booleans to integers
        for col in selected_features:

            if pd.api.types.is_bool_dtype(
                cox_data[col]
            ):

                cox_data[col] = (
                    cox_data[col]
                    .astype(int)
                )

            else:

                cox_data[col] = (
                    pd.to_numeric(
                        cox_data[col],
                        errors="coerce"
                    )
                )

        # Remove constant columns
        usable_features = [
            col
            for col in selected_features
            if cox_data[col].nunique(
                dropna=True
            ) > 1
        ]

        cox_data = cox_data[
            [duration_col, event_col]
            + usable_features
        ]

        # Median imputation
        for col in usable_features:

            cox_data[col] = (
                cox_data[col]
                .fillna(
                    cox_data[col].median()
                )
            )

        # ----------------------------------------------
        # Standardize continuous/non-binary predictors
        # ----------------------------------------------

        for col in usable_features:

            if cox_data[col].nunique() > 2:

                std = (
                    cox_data[col].std()
                )

                if std > 0:

                    cox_data[col] = (
                        cox_data[col]
                        - cox_data[col].mean()
                    ) / std

        try:

            multivariate_cox = (
                CoxPHFitter(
                    penalizer=penalizer
                )
            )

            multivariate_cox.fit(
                cox_data,
                duration_col=duration_col,
                event_col=event_col
            )

            multivariate_summary = (
                multivariate_cox
                .summary
                .copy()
            )

            print(
                "\n=== Multivariate Cox-PH ==="
            )

            print(
                multivariate_summary[
                    [
                        "coef",
                        "exp(coef)",
                        "p"
                    ]
                ]
                .sort_values("p")
            )

            print(
                "\nConcordance index:",
                multivariate_cox
                .concordance_index_
            )

            # ==========================================
            # 8. PH assumption test
            # ==========================================

            ph_test = (
                proportional_hazard_test(
                    multivariate_cox,
                    cox_data,
                    time_transform="rank"
                )
            )

            ph_test_df = (
                ph_test.summary.copy()
            )

            print(
                "\n=== Proportional-hazards test ==="
            )

            print(
                ph_test_df.sort_values(
                    "p"
                )
            )

        except Exception as e:

            print(
                "\nMultivariate Cox model failed:"
            )

            print(e)

    # ==========================================================
    # Results
    # ==========================================================

    return {
        "dataset_summary":
            dataset_summary,

        "kaplan_meier":
            kmf,

        "km_group_results":
            km_group_results,

        "univariate_cox":
            univariate_cox,

        "multivariate_cox":
            multivariate_cox,

        "multivariate_summary":
            multivariate_summary,

        "ph_test":
            ph_test_df
    }

In [ ]:
exclude_columns = [
    # Identifier
    "id",

    # Survival outcome
    "remainingDays",
    "totalDays"
    "eventHasHappend",
	"totalWorkDays",
	"totalRestDays",

    # Future information / leakage
    "targetEndDate",
    "assignmentsAfterCut",
    "ausgesch-am",

    # Dates
    "cutDate",
    "startofCaregiver",
    "endofCaregiver",
    "eingestellt-am",

    # Depending on whether these are still present
    "startDate",
    "endDate"
]

all_features = (
    df
    .select_dtypes(include=["number", "bool"])
    .columns
    .difference(exclude_columns)
    .tolist()
)

print("Number of features:", len(all_features))


results = base_survival_analysis(
    df,
    duration_col="totalDays",
    event_col="eventHasHappend",
    feature_columns=all_features,
    km_group_column="isPaymentBlocked",
    show_plots=True,
    penalizer=0.01
)

In [ ]:
results

In [ ]:
pd.set_option('display.max_rows', 500)

In [ ]:
univariate = results["univariate_cox"]

univariate[
    [
        "feature",
        "hazard_ratio",
        "ci_lower",
        "ci_upper",
        "p"
    ]
].sort_values("hazard_ratio")